# Kaggle Competition Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forge-features/forge/blob/main/notebooks/05_kaggle_workflow.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/forge-features/forge/main?labpath=notebooks/05_kaggle_workflow.ipynb)

This notebook demonstrates a complete Kaggle-style workflow using Forge.

## What you'll learn

1. Loading and exploring data
2. Automatic data analysis
3. Feature engineering strategy
4. Model training and evaluation
5. Creating submissions

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report

np.random.seed(42)

## Step 1: Create Sample Competition Data

We'll simulate a typical Kaggle binary classification problem:

In [ ]:
# Simulate a customer churn prediction dataset
n_train = 5000
n_test = 2000

def generate_data(n_samples):
    data = {
        # Numeric features
        'age': np.random.randint(18, 70, n_samples),
        'tenure_months': np.random.exponential(24, n_samples).astype(int),
        'monthly_charges': np.random.lognormal(4, 0.5, n_samples),
        'total_charges': np.random.lognormal(7, 1, n_samples),
        'num_products': np.random.poisson(2, n_samples) + 1,
        'support_tickets': np.random.poisson(1.5, n_samples),
        
        # Categorical features
        'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
        'payment_method': np.random.choice(['Credit card', 'Bank transfer', 'Electronic check', 'Mailed check'], n_samples),
        'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples),
        'gender': np.random.choice(['Male', 'Female'], n_samples),
        
        # Date feature
        'signup_date': pd.date_range('2020-01-01', periods=n_samples, freq='H')[:n_samples],
    }
    return pd.DataFrame(data)

# Generate train and test data
train_df = generate_data(n_train)
test_df = generate_data(n_test)

# Create target (churn) based on some logic
churn_prob = (
    0.1 +
    0.3 * (train_df['contract_type'] == 'Month-to-month').astype(float) +
    0.2 * (train_df['support_tickets'] > 2).astype(float) +
    0.1 * (train_df['tenure_months'] < 12).astype(float) -
    0.1 * (train_df['num_products'] > 2).astype(float)
)
train_df['churn'] = (np.random.random(n_train) < churn_prob).astype(int)

print(f"Training data: {train_df.shape}")
print(f"Test data: {test_df.shape}")
print(f"\nChurn rate: {train_df['churn'].mean():.2%}")

## Step 2: Data Exploration with Forge Analyzer

In [ ]:
from forge import DataAnalyzer

# Separate features and target
X_train = train_df.drop('churn', axis=1)
y_train = train_df['churn']
X_test = test_df.copy()

# Analyze the data
analyzer = DataAnalyzer()
report = analyzer.analyze(X_train, y_train)

print("=" * 50)
print("DATA ANALYSIS REPORT")
print("=" * 50)

print("\nColumn Types:")
for col, dtype in report.column_types.items():
    print(f"  {col}: {dtype}")

print("\nQuality Issues:")
if report.quality_issues:
    for issue in report.quality_issues:
        print(f"  - {issue}")
else:
    print("  No quality issues found!")

## Step 3: Feature Engineering with AutoFeatureTransformer

In [ ]:
from forge import AutoFeatureTransformer

# Create transformer with competition-oriented settings
transformer = AutoFeatureTransformer(
    max_features=100,           # Keep top 100 features
    selection_method='importance',  # Use tree-based importance
    n_jobs=-1,                  # Use all CPU cores
    verbose=1
)

# Fit on training data
X_train_fe = transformer.fit_transform(X_train, y_train)

# Transform test data (using fitted transformer)
X_test_fe = transformer.transform(X_test)

print(f"\nOriginal features: {X_train.shape[1]}")
print(f"Engineered features: {X_train_fe.shape[1]}")

## Step 4: Feature Importance Analysis

In [ ]:
# Get feature importance
importance = transformer.get_feature_importance()

print("Top 15 Most Important Features:")
print("=" * 50)
for i, (feature, score) in enumerate(importance.head(15).items(), 1):
    print(f"{i:2d}. {feature:40s} {score:.4f}")

## Step 5: Model Training and Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Use stratified k-fold for imbalanced data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Train Gradient Boosting classifier
model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

# Cross-validation
cv_scores = cross_val_score(
    model, 
    X_train_fe, 
    y_train, 
    cv=cv, 
    scoring='roc_auc'
)

print(f"Cross-validation ROC-AUC scores: {cv_scores}")
print(f"Mean ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## Step 6: Final Model Training

In [ ]:
# Train on full training data
model.fit(X_train_fe, y_train)

# Get predictions on training data for sanity check
train_pred_proba = model.predict_proba(X_train_fe)[:, 1]
train_auc = roc_auc_score(y_train, train_pred_proba)

print(f"Training ROC-AUC: {train_auc:.4f}")

## Step 7: Generate Predictions for Test Set

In [ ]:
# Generate predictions
test_predictions = model.predict_proba(X_test_fe)[:, 1]

# Create submission file
submission = pd.DataFrame({
    'id': range(len(test_predictions)),
    'churn_probability': test_predictions
})

print("Submission preview:")
print(submission.head(10))

print(f"\nPrediction statistics:")
print(f"  Min: {test_predictions.min():.4f}")
print(f"  Max: {test_predictions.max():.4f}")
print(f"  Mean: {test_predictions.mean():.4f}")

In [ ]:
# Save submission
submission.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'")

## Step 8: Complete Pipeline for Production

In [ ]:
from sklearn.pipeline import Pipeline
import joblib

# Create a complete pipeline
complete_pipeline = Pipeline([
    ('feature_engineering', AutoFeatureTransformer(max_features=100)),
    ('classifier', GradientBoostingClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train the complete pipeline
complete_pipeline.fit(X_train, y_train)

# Save for production
joblib.dump(complete_pipeline, 'churn_model.joblib')
print("Complete pipeline saved to 'churn_model.joblib'")

# Example: Loading and using in production
loaded_model = joblib.load('churn_model.joblib')
new_predictions = loaded_model.predict_proba(X_test)[:, 1]
print(f"\nLoaded model predictions match: {np.allclose(new_predictions, test_predictions)}")

In [ ]:
# Clean up
import os
for f in ['submission.csv', 'churn_model.joblib']:
    if os.path.exists(f):
        os.remove(f)
print("Cleaned up temporary files")

## Summary

In this notebook, we demonstrated a complete Kaggle workflow:

1. **Data Exploration**: Used `DataAnalyzer` to understand the data
2. **Feature Engineering**: Used `AutoFeatureTransformer` to automatically generate features
3. **Model Training**: Trained a Gradient Boosting classifier
4. **Validation**: Used stratified cross-validation
5. **Predictions**: Generated test predictions
6. **Production**: Created a complete pipeline for deployment

Forge reduced the feature engineering effort significantly while maintaining interpretability and sklearn compatibility.